In [2]:
from __future__ import annotations

import re
import unicodedata
from collections import Counter
from typing import Dict, Iterable, List, Sequence, Tuple
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Supaya output dataframe tidak terpotong
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Soal 01

## 1. Mengapa `analisis-data` dan `analisis data` bisa menghasilkan fitur yang berbeda?

analisis-data dan analisis data dapat menghasilkan fitur yang berbeda karena pada proses preprocessing NLP, tanda hubung (`-`) dan spasi diperlakukan berbeda oleh tokenizer.

Pada kata **“analisis-data”**, sistem biasanya menganggapnya sebagai **satu token** sehingga fitur yang terbentuk adalah:

```text
analisis-data
```

Sedangkan pada **“analisis data”**, sistem memisahkan berdasarkan spasi sehingga menjadi dua token:

```text
analisis
data
```

Akibatnya, representasi fitur seperti Bag of Words, TF-IDF, maupun embedding akan berbeda karena model menganggap kedua bentuk tersebut bukan kata yang sama.

Perbedaan ini terjadi karena proses tokenisasi sangat dipengaruhi oleh:

* tanda baca,
* spasi,
* aturan preprocessing,
* dan metode tokenizer yang digunakan.

Oleh karena itu, dalam NLP biasanya dilakukan normalisasi teks seperti menghapus tanda baca atau menyamakan format penulisan agar fitur yang dihasilkan lebih konsisten.


In [3]:
# Implementasi
# fungsi tokenisasi berbasis regex 
def regex_word_tokenizer(text: str) -> List[str]:
    return re.findall(r"[A-Za-zÀ-ÿ0-9_]+(?:-[A-Za-z]+)?", text) 

# dua kalimat yang dibandingkan
text1 = "analisis-data"
text2 = "analisis data"

# tokenisasi
tokens1 = regex_word_tokenizer(text1)
tokens2 = regex_word_tokenizer(text2)

print("Token teks 1:", tokens1)
print("Token teks 2:", tokens2)

# representasi fitur sederhana (Bag of Words)
bow1 = Counter(tokens1)
bow2 = Counter(tokens2)

print("\nFitur teks 1:", bow1)
print("Fitur teks 2:", bow2)

Token teks 1: ['analisis-data']
Token teks 2: ['analisis', 'data']

Fitur teks 1: Counter({'analisis-data': 1})
Fitur teks 2: Counter({'analisis': 1, 'data': 1})


## 2. Kapan hashtag sebaiknya dipertahankan, dan kapan boleh dihapus?

Hashtag (`#`) dalam teks media sosial dapat diperlakukan berbeda tergantung pada tujuan analisis dalam Natural Language Processing (NLP).

**1. Hashtag sebaiknya dipertahankan**: Ketika memiliki informasi penting yang berhubungan dengan makna teks atau tujuan analisis. Dalam banyak kasus media sosial, hashtag berfungsi sebagai penanda topik, tren, atau kategori tertentu.

Contoh:

* `#covid19`
* `#Pemilu2024`
* `#AI`

Hashtag seperti ini sebaiknya dipertahankan karena dapat menjadi fitur penting dalam:

* Analisis sentimen
* Topic modeling
* Trend analysis
* Event detection

Biasanya hashtag tidak dihapus sepenuhnya, tetapi tanda `#` dihilangkan sehingga menjadi token seperti “covid19” atau “pemilu2024”.

---

**2. Hashtag boleh dihapus**: Jika tidak memberikan informasi penting atau dianggap sebagai noise dalam proses analisis. Ini biasanya terjadi pada analisis teks umum yang tidak berfokus pada media sosial.

Contoh:

* “Saya belajar NLP #belajar”

Jika hashtag tidak relevan dengan tujuan analisis, maka dapat dihapus sehingga hanya tersisa:

* “Saya belajar NLP”


In [19]:
# Memisahkan hashtag CamelCase menjadi beberapa kata
def split_hashtag_camel(tag):
    words = re.sub(r"([A-Z])", r" \1", tag).strip().split()
    return [w.lower() for w in words]

# Menangani hashtag
def handle_hashtag(text, mode="extract"):
    if mode == "remove":
        # Menghapus hashtag sepenuhnya
        return re.sub(r"#\w+", "", text)
    
    elif mode == "extract":
        # Mempertahankan isi hashtag tanpa tanda #
        return re.sub(r"#(\w+)", r"\1", text)


# 1. Hashtag dipertahankan

text1 = "Topik penelitian hari ini adalah #MachineLearning dan #DataScience"

print("Contoh hashtag dipertahankan:")
print(handle_hashtag(text1, mode="extract"))

# Memisahkan hashtag CamelCase
print(split_hashtag_camel("MachineLearning"))
print(split_hashtag_camel("DataScience"))

# 2. Hashtag dihapus

text2 = "Saya sedang belajar NLP #belajar #semangat"

print("\nContoh hashtag dihapus:")
print(handle_hashtag(text2, mode="remove"))

Contoh hashtag dipertahankan:
Topik penelitian hari ini adalah MachineLearning dan DataScience
['machine', 'learning']
['data', 'science']

Contoh hashtag dihapus:
Saya sedang belajar NLP  


Penjelasan:

* Pada contoh pertama, hashtag dipertahankan karena mengandung topik penting (`#MachineLearning`, `#DataScience`).
* Pada contoh kedua, hashtag dihapus karena dianggap tidak menambah informasi penting (`#belajar`, `#semangat`).

## 3. Mengapa lowercasing bisa membantu pada satu task tetapi merugikan pada task lain?

Lowercasing dalam Natural Language Processing (NLP) digunakan untuk mengubah seluruh teks menjadi huruf kecil sehingga variasi penulisan huruf besar tidak dianggap sebagai token yang berbeda. Proses ini dapat memberikan dampak yang berbeda tergantung pada jenis tugas (task) yang dikerjakan, karena lowercasing dapat menghilangkan informasi kapitalisasi yang dalam beberapa kasus memiliki makna penting.

Lowercasing umumnya membantu pada task seperti Information Retrieval (IR) dan topic modeling, karena pada tugas tersebut perbedaan huruf besar dan kecil tidak membawa perbedaan makna yang signifikan. Dengan melakukan lowercasing, variasi kata dapat dikurangi sehingga representasi kata menjadi lebih konsisten, dimensi fitur menjadi lebih kecil, dan proses pemodelan menjadi lebih efisien.

Namun, lowercasing perlu dipertimbangkan dengan hati-hati pada task yang sensitif terhadap kapitalisasi seperti Named Entity Recognition (NER), information extraction, atau case-sensitive text processing. Pada tugas-tugas tersebut, kapitalisasi dapat mengandung informasi semantik penting yang berhubungan dengan identitas atau kategori tertentu.

Dengan demikian, lowercasing merupakan proses yang bersifat trade-off, yaitu dapat meningkatkan efisiensi dan konsistensi representasi data pada beberapa task, tetapi juga dapat mengurangi informasi penting pada task lain yang bergantung pada kapitalisasi.

Contoh:

| Situasi | Contoh Input | Hasil Lowercasing | Alasan |
|---------|--------|---------|--------|
| Information Retrieval / Topic Modeling (membantu) | "Data Mining", "DATA MINING" | "data mining", "data mining" | Menyatukan variasi penulisan sehingga fitur lebih konsisten dan mengurangi jumlah vocabulary |
| Text Classification umum (membantu) | "Saya suka AI", "SAYA SUKA ai" | "saya suka ai", "saya suka ai" | Mengurangi noise variasi huruf besar kecil tanpa mengubah makna utama kalimat |
| Named Entity Recognition (perlu dipertimbangkan) | "Budi Santoso", "budi santoso" | "budi santoso", "budi santoso" | Kehilangan sinyal bahwa ini adalah nama orang (entitas khusus)
| Case-sensitive word meaning (perlu dipertimbangkan) | "US", "us" | "us", "us" | Menghilangkan perbedaan antara negara (United States) dan kata ganti “us” |
| Sentiment / Emphasis detection (perlu dipertimbangkan) | "INI LUAR BIASA", "ini luar biasa" | "ini luar biasa", "ini luar biasa" | Hilangnya informasi penekanan emosi yang biasanya ditandai kapitalisasi |

## 4. Apa perbedaan dampak tokenization pada sentiment analysis dan log analysis?

Pada sentiment analysis, tokenization memengaruhi kemampuan model dalam memahami makna bahasa. Token yang tidak tepat dapat mengubah interpretasi sentimen suatu kalimat. Misalnya, kalimat “tidak bagus” harus dipisahkan menjadi dua token, yaitu “tidak” dan “bagus”. Kata “tidak” berfungsi sebagai negasi yang mengubah makna kata “bagus” menjadi sentimen negatif. Jika tokenization gagal memisahkan kedua kata tersebut, model dapat salah memahami sentimen kalimat.

Selain itu, sentiment analysis sering menangani emoji, slang, singkatan, dan tanda baca yang memiliki nilai emosional. Contohnya, emoji marah atau kata seperti “bagusss” dapat menjadi indikator sentimen yang penting. Oleh karena itu, tokenization pada sentiment analysis harus mampu mempertahankan konteks linguistik dan makna emosional teks.

Sebaliknya, pada log analysis, tokenization lebih berfokus pada struktur teknis dibandingkan makna bahasa. Data log biasanya berisi timestamp, alamat IP, kode error, nama proses, dan status sistem. Contohnya:

```python 
ERROR 2026-05-19 code=500 ip=192.168.1.1
```

Dalam kasus ini, token seperti `code=500` atau `192.168.1.1` harus dipertahankan secara utuh karena mengandung informasi penting untuk mendeteksi error atau anomali sistem. Jika tokenization memecah IP address atau kode error secara tidak tepat, informasi teknis dapat hilang dan analisis menjadi tidak akurat.

## 5. Mengapa bahasa Indonesia membutuhkan perhatian khusus dibanding sekadar menerapkan tokenizer bahasa Inggris?

Bahasa Indonesia membutuhkan perhatian khusus dalam proses tokenization karena memiliki struktur bahasa, bentuk kata, dan pola penulisan yang berbeda dari bahasa Inggris. Jika tokenizer bahasa Inggris digunakan secara langsung, hasil tokenisasi dapat menjadi kurang akurat dan memengaruhi kualitas analisis NLP.

Salah satu perbedaannya adalah penggunaan imbuhan. Bahasa Indonesia memiliki banyak prefiks, sufiks, konfiks, dan reduplikasi. Contohnya adalah kata “membacakan”, “berjualan”, atau “menuliskan”. Kata-kata tersebut terbentuk dari kombinasi kata dasar dan imbuhan yang mengubah makna maupun fungsi kata. Tokenizer bahasa Inggris umumnya tidak dirancang untuk menangani pola morfologi seperti ini.

Bahasa Indonesia juga sering menggunakan kata ulang, seperti “anak-anak”, “buku-buku”, atau “jalan-jalan”. Dalam beberapa konteks, tanda hubung memiliki makna penting dan tidak selalu boleh dipisahkan. Jika tokenizer bahasa Inggris memecah kata tersebut secara sembarangan, makna asli kata dapat berubah.

Selain itu, bahasa Indonesia memiliki banyak variasi informal dan slang, terutama pada media sosial. Contohnya seperti:

* “gk”
* “gak”
* “nggak”
* “ga”
* “tdk”

Semua kata tersebut dapat memiliki makna yang sama, yaitu “tidak”. Tokenizer bahasa Inggris biasanya tidak memiliki aturan atau kamus untuk memahami variasi bahasa Indonesia seperti ini.

Bahasa Indonesia juga sering mencampurkan bahasa asing dan singkatan dalam satu kalimat. Contohnya:

* “Aku lagi meeting sama tim”
* “Tugasnya udh di-submit belum?”

Kondisi ini membuat tokenization menjadi lebih kompleks karena sistem harus mampu mengenali campuran bahasa dan bentuk tidak baku.

Karena itu, NLP bahasa Indonesia membutuhkan tokenizer yang dirancang khusus atau telah disesuaikan dengan karakteristik bahasa Indonesia agar hasil pemrosesan teks menjadi lebih akurat dan relevan.


## 6. Apa perbedaan intrinsic evaluation dan extrinsic evaluation pada tokenization?

### Intrinsic Evaluation

Intrinsic evaluation menilai kualitas tokenization secara langsung berdasarkan hasil token yang dihasilkan. Evaluasi ini berfokus pada seberapa tepat tokenizer memecah teks sesuai standar atau anotasi yang benar.

Pada pendekatan ini, hasil tokenization dibandingkan dengan data referensi atau gold standard. Jika token yang dihasilkan sesuai dengan token yang diharapkan, maka tokenizer dianggap memiliki performa yang baik.

Contoh:

Kalimat:

```python id="9gfl1n"
"Saya suka machine-learning"
```

Hasil tokenisasi yang benar:

```python id="i5jlwm"
["Saya", "suka", "machine-learning"]
```

Jika tokenizer memecah menjadi:

```python id="ch8lqy"
["Saya", "suka", "machine", "learning"]
```

maka hasil tersebut dapat dianggap kurang tepat tergantung aturan tokenisasi yang digunakan.

Intrinsic evaluation biasanya menggunakan metrik seperti:

* Accuracy
* Precision
* Recall
* F1-score

---

### Extrinsic Evaluation

Extrinsic evaluation menilai tokenizer berdasarkan dampaknya pada task lain. Evaluasi ini tidak hanya melihat hasil tokenisasi, tetapi juga melihat apakah tokenization membantu meningkatkan performa model akhir.

Contoh task:

* Sentiment analysis
* Machine translation
* Named Entity Recognition
* Text classification

Misalnya, dua tokenizer menghasilkan token berbeda. Kemudian kedua hasil tokenisasi digunakan pada model sentiment analysis. Tokenizer yang menghasilkan akurasi klasifikasi lebih tinggi dianggap lebih baik secara extrinsic evaluation.

Contoh:

* Tokenizer A → akurasi sentiment analysis 82%
* Tokenizer B → akurasi sentiment analysis 90%

Maka tokenizer B dianggap lebih efektif untuk task tersebut.

# Soal 02

## 1. Jelaskan mengapa stopword removal tidak selalu cocok untuk sentiment analysis.

Stopword removal tidak selalu cocok untuk sentiment analysis karena beberapa stopword justru membawa informasi sentimen yang penting. Dalam sentiment analysis, makna kalimat tidak hanya ditentukan oleh kata utama, tetapi juga oleh kata fungsi seperti negasi.

Pada materi preprocessing NLP, dijelaskan bahwa kata negasi seperti 'tidak', 'bukan', 'belum', dan 'jangan' sering menjadi inti makna kalimat. Jika kata-kata tersebut dihapus, polaritas sentimen dapat berubah total. 

Contoh:

* “produk ini bagus” → sentimen positif
* “produk ini tidak bagus” → sentimen negatif

Jika stopword removal dilakukan terlalu agresif dan kata'tidak' dihapus, maka kalimat “produk ini tidak bagus” berubah menjadi “produk ini bagus”. Akibatnya, model dapat salah mengklasifikasikan sentimen. 

Selain negasi, beberapa kata fungsi juga membantu model memahami konteks opini, intensitas, dan hubungan antarkata. Karena itu, stopword removal harus bersifat *task-aware* dan *domain-aware*. Pada sentiment analysis, preprocessing biasanya dilakukan secara selektif agar informasi penting tetap dipertahankan. 

Sebagai alternatif, materi juga menjelaskan penggunaan *negation marking*. Teknik ini tidak menghapus kata negasi, tetapi menandai kata setelah negasi, misalnya:

* “tidak bagus” → “tidak bagus_NEG”

Strategi ini membantu model membedakan kata “bagus” dalam konteks positif dan negatif. 


In [20]:
# Implementasi
# Tokenizer sederhana

def simple_tokenize(text: str) -> List[str]:
    text = text.lower()
    return re.findall(r"[a-zA-ZÀ-ÿ0-9_]+", text)

# Stopword dan negasi

STOPWORDS_ID = {
    "yang", "dan", "di", "ke", "dari", "untuk", "pada", "dengan", "adalah", "itu", "ini",
    "atau", "sebagai", "karena", "dalam", "sedang", "saya", "kami", "kita", "anda",
    "para", "suatu", "sebuah", "oleh", "juga", "bahwa", "agar", "dapat", "akan",
    "sangat", "lebih"
}

NEGATION_WORDS = {"tidak", "bukan", "belum", "jangan", "tak", "gak", "ga"}

# Fungsi stopword removal

def remove_stopwords(tokens: List[str], stopwords: set[str]) -> List[str]:
    return [t for t in tokens if t not in stopwords]

# Data contoh sentiment analysis

sentences = [
    "Produk ini bagus",
    "Produk ini tidak bagus",
    "Layanan ini sangat cepat",
    "Layanan ini tidak cepat",
    "Model ini belum stabil"
]

rows = []

for sentence in sentences:

    # Tokenisasi
    tokens = simple_tokenize(sentence)

    # Stopword removal aman
    # Negasi dipertahankan
    safe = remove_stopwords(tokens, STOPWORDS_ID)

    # Stopword removal terlalu agresif
    # Negasi ikut dihapus
    unsafe = remove_stopwords(
        tokens,
        STOPWORDS_ID | NEGATION_WORDS
    )

    rows.append([
        sentence,
        tokens,
        safe,
        unsafe
    ])


# Tampilkan hasil

df = pd.DataFrame(
    rows,
    columns=[
        "Kalimat",
        "Token Awal",
        "Stopword Removal Aman",
        "Terlalu Agresif"
    ]
)

print(df)

                    Kalimat                     Token Awal  \
0          Produk ini bagus           [produk, ini, bagus]   
1    Produk ini tidak bagus    [produk, ini, tidak, bagus]   
2  Layanan ini sangat cepat  [layanan, ini, sangat, cepat]   
3   Layanan ini tidak cepat   [layanan, ini, tidak, cepat]   
4    Model ini belum stabil    [model, ini, belum, stabil]   

     Stopword Removal Aman   Terlalu Agresif  
0          [produk, bagus]   [produk, bagus]  
1   [produk, tidak, bagus]   [produk, bagus]  
2         [layanan, cepat]  [layanan, cepat]  
3  [layanan, tidak, cepat]  [layanan, cepat]  
4   [model, belum, stabil]   [model, stabil]  


Hasil preprocessing menunjukkan bahwa stopword removal dapat mengubah makna sentimen jika dilakukan terlalu agresif.

Pada kalimat tanpa negasi, seperti “Produk ini bagus”, makna tetap sama setelah preprocessing. Namun, pada kalimat seperti “Produk ini tidak bagus” dan “Layanan ini tidak cepat”, penghapusan kata negasi `tidak` menyebabkan sentimen negatif berubah menjadi positif.

Hal yang sama terjadi pada kata `belum` dalam kalimat “Model ini belum stabil”. Jika kata tersebut dihapus, makna kalimat berubah seolah-olah model sudah stabil.

Karena itu, pada sentiment analysis, kata negasi sebaiknya dipertahankan agar informasi sentimen tidak hilang.

## 2. Berikan tiga contoh kalimat yang maknanya berubah jika kata negasi dihapus.

In [24]:
# Contoh kalimat
sentences = [
    "Saya tidak setuju dengan keputusan tersebut",
    "Produk ini bukan pilihan terbaik",
    "Model ini belum stabil untuk digunakan"
]

# Proses preprocessing
rows = []

for sentence in sentences:
    
    # Tokenisasi
    tokens = simple_tokenize(sentence)
    
    # Preprocessing terlalu agresif
    preprocessing = remove_stopwords(
        tokens,
        NEGATION_WORDS
    )
    
    rows.append([
        sentence,
        tokens,
        " ".join(preprocessing)
    ])

# DataFrame hasil
df = pd.DataFrame(
    rows,
    columns=[
        "Kalimat Asli",
        "Tokens",
        "Hasil Preprocessing"
    ]
)

print(df)

                                  Kalimat Asli  \
0  Saya tidak setuju dengan keputusan tersebut   
1             Produk ini bukan pilihan terbaik   
2       Model ini belum stabil untuk digunakan   

                                               Tokens  \
0  [saya, tidak, setuju, dengan, keputusan, tersebut]   
1              [produk, ini, bukan, pilihan, terbaik]   
2       [model, ini, belum, stabil, untuk, digunakan]   

                     Hasil Preprocessing  
0  saya setuju dengan keputusan tersebut  
1             produk ini pilihan terbaik  
2       model ini stabil untuk digunakan  


Berikut tiga contoh kalimat yang maknanya berubah jika kata negasi dihapus.

1. “Saya tidak setuju dengan keputusan tersebut.”

   * Setelah kata negasi dihapus: “Saya setuju dengan keputusan tersebut.”
   * Makna berubah dari penolakan menjadi persetujuan.

2. “Produk ini bukan pilihan terbaik.”

   * Setelah kata negasi dihapus: “Produk ini pilihan terbaik.”
   * Makna berubah dari kritik menjadi pujian.

3. “Model ini belum stabil untuk digunakan.”

   * Setelah kata negasi dihapus: “Model ini stabil untuk digunakan.”
   * Makna berubah dari kondisi belum siap menjadi siap digunakan.

Dalam materi preprocessing NLP, dijelaskan bahwa kata negasi seperti 'tidak', 'bukan', 'belum', dan 'jangan' sering menjadi inti makna pada sentiment analysis. Jika kata tersebut dihapus secara agresif, polaritas dan arti kalimat dapat berubah total. 

## 3. Bandingkan hasil stemming untuk kata pemerintah, perintah, memerintah, dan pemerintahan.

In [6]:
# Cell ini dibuat robust. Jika Sastrawi belum terpasang, notebook tetap bisa berjalan memakai fallback sederhana.
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
    STEMMER_NAME = "Sastrawi"
except Exception:
    class SimpleIndonesianStemmer:
        def stem(self, word: str) -> str:
            # Fallback pedagogis, bukan stemmer produksi.
            rules = [
                (r"^meng", ""), (r"^meny", "s"), (r"^men", ""), (r"^mem", ""), (r"^me", ""),
                (r"^peng", ""), (r"^peny", "s"), (r"^pen", ""), (r"^pem", ""),
                (r"^ber", ""), (r"^per", ""), (r"^di", ""), (r"^ter", ""),
                (r"kan$", ""), (r"an$", ""), (r"i$", ""),
            ]
            stem = word
            for pattern, repl in rules:
                stem = re.sub(pattern, repl, stem)
            return stem
    stemmer = SimpleIndonesianStemmer()
    STEMMER_NAME = "fallback sederhana"

print("Stemmer aktif:", STEMMER_NAME)

words = ["pemerintah", "perintah", "memerintah", "pemerintahan"]
pd.DataFrame([[w, stemmer.stem(w)] for w in words], columns=["kata", "hasil_stemming"])

Stemmer aktif: Sastrawi


,kata,hasil_stemming
0,pemerintah,perintah
1,perintah,perintah
2,memerintah,perintah
3,pemerintahan,perintah


Perbandingan hasil stemming untuk kata *pemerintah*, *perintah*, *memerintah*, dan *pemerintahan* menunjukkan bahwa stemming dapat menyatukan beberapa kata ke bentuk akar yang sama, tetapi juga berisiko menghilangkan perbedaan makna.

| Kata         | Hasil stemming | Penjelasan                               |
| ------------ | -------------- | ---------------------------------------- |
| pemerintah   | perintah       | imbuhan *pe-* dianggap afiks dan dihapus |
| perintah     | perintah       | sudah dianggap bentuk dasar              |
| memerintah   | perintah       | prefiks *me-* dihapus                    |
| pemerintahan | perintah       | prefiks *pe-* dan sufiks *-an* dihapus   |

Dari hasil tersebut terlihat bahwa tiga kata berbeda, yaitu *pemerintah*, *memerintah*, dan *pemerintahan*, dapat direduksi menjadi bentuk yang sama, yaitu *perintah*.

Hal ini menunjukkan salah satu risiko stemming, yaitu *over-stemming*. Dalam materi dijelaskan bahwa over-stemming terjadi ketika beberapa kata yang memiliki makna berbeda disatukan menjadi satu bentuk akar. 

Secara semantik:

* *pemerintah* mengacu pada institusi atau pihak pengelola negara,
* *memerintah* adalah aktivitas memberi perintah,
* *pemerintahan* merujuk pada sistem atau proses pemerintahan,
* sedangkan *perintah* berarti instruksi atau komando.

Jika semua kata tersebut direduksi menjadi *perintah*, sebagian informasi makna dapat hilang. Karena itu, stemming perlu digunakan secara hati-hati dan disesuaikan dengan tujuan analisis NLP.


## 4. Buat pipeline preprocessing untuk data tweet bahasa Indonesia.

Pipeline preprocessing untuk data tweet bahasa Indonesia harus disesuaikan dengan karakteristik media sosial. Tweet biasanya mengandung singkatan, hashtag, mention, URL, emoji, kata tidak baku, dan noise. Oleh karena itu, preprocessing perlu dilakukan secara bertahap agar informasi penting tetap terjaga.

Berikut contoh pipeline preprocessing untuk tweet bahasa Indonesia.

1. **Case Folding**
   Mengubah seluruh huruf menjadi lowercase agar token konsisten.
   Contoh:

   * “Produk Ini BAGUS” → “produk ini bagus”

2. **Cleaning Text**
   Menghapus elemen yang tidak relevan seperti:

   * URL
   * mention (@username)
   * hashtag simbol (#)
   * tanda baca berlebih
   * karakter berulang

   Contoh:

   * “Keren bangettt!!! 😍 #AI”
     → “keren banget ai”

3. **Tokenization**
   Memecah kalimat menjadi token/kata.

   Contoh:

   * “sistem tidak stabil”
     → [“sistem”, “tidak”, “stabil”]

4. **Normalization**
   Mengubah kata slang atau tidak baku menjadi bentuk standar.

   Contoh:

   * “gk”, “ga”, “gak” → “tidak”
   * “bgt” → “banget”

5. **Negation Handling**
   Kata negasi tidak boleh dihapus sembarangan karena dapat mengubah sentimen. Materi preprocessing menjelaskan bahwa kata seperti 'tidak', 'bukan', dan 'belum' merupakan inti makna dalam sentiment analysis. 

   Contoh:

   * “tidak bagus”
     → “tidak bagus_NEG”

6. **Selective Stopword Removal**
   Menghapus stopword yang tidak penting, tetapi mempertahankan kata negasi.

   Contoh:

   * hapus: “dan”, “yang”, “di”
   * pertahankan: “tidak”, “bukan”

7. **Stemming atau Lemmatization**
   Mengubah kata berimbuhan menjadi bentuk dasar untuk mengurangi variasi morfologis. 

   Contoh:

   * “membaca” → “baca”
   * “pengembangan” → “kembang”

8. **Feature Extraction**
   Hasil preprocessing digunakan untuk representasi fitur seperti:

   * Bag of Words
   * n-gram
   * TF
   * TF-IDF

   Materi menjelaskan bahwa preprocessing menjadi dasar sebelum representasi fitur dilakukan. 


In [25]:
# DATA AWAL

tweet = "Produk ini gak bagus!!! Sistemnya sering error 😭 #kecewa"

print("Tweet awal:")
print(tweet)

# 1. CASE FOLDING

text = tweet.lower()

print("\n1. Case Folding:")
print(text)

# 2. CLEANING TEXT

# hapus URL
text = re.sub(r"http\S+|www\S+", "", text)

# hapus mention
text = re.sub(r"@\w+", "", text)

# hapus hashtag simbol (#)
text = re.sub(r"#", "", text)

# hapus emoji dan karakter non huruf
text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

# hapus spasi berlebih
text = re.sub(r"\s+", " ", text).strip()

print("\n2. Cleaning Text:")
print(text)

# 3. TOKENIZATION

tokens = text.split()

print("\n3. Tokenization:")
print(tokens)

# 4. NORMALIZATION

normalization_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "gak": "tidak",
    "bgt": "banget"
}

normalized_tokens = []

for token in tokens:
    normalized_tokens.append(
        normalization_dict.get(token, token)
    )

print("\n4. Normalization:")
print(normalized_tokens)

# 5. NEGATION HANDLING

negation_words = {"tidak", "bukan", "belum", "jangan"}

marked_tokens = []

for i, token in enumerate(normalized_tokens):

    if token in negation_words:
        marked_tokens.append(token)

        # tandai kata setelah negasi
        if i + 1 < len(normalized_tokens):
            next_token = normalized_tokens[i + 1]
            marked_tokens.append(next_token + "_NEG")

    elif i > 0 and normalized_tokens[i - 1] in negation_words:
        continue

    else:
        marked_tokens.append(token)

print("\n5. Negation Handling:")
print(marked_tokens)

# 6. SELECTIVE STOPWORD REMOVAL

stopwords = {
    "ini", "yang", "dan", "di", "ke", "dari"
}

filtered_tokens = [
    token for token in marked_tokens
    if token not in stopwords
]

print("\n6. Stopword Removal:")
print(filtered_tokens)

# 7. STEMMING SEDERHANA

def simple_stemming(word):

    suffixes = ["nya", "kan", "an"]

    for suffix in suffixes:
        if word.endswith(suffix):
            word = word[:-len(suffix)]

    return word

stemmed_tokens = []

for token in filtered_tokens:

    if token.endswith("_NEG"):

        base_word = token.replace("_NEG", "")
        stemmed_word = simple_stemming(base_word)
        stemmed_tokens.append(stemmed_word + "_NEG")

    else:
        stemmed_tokens.append(
            simple_stemming(token)
        )

print("\n7. Stemming:")
print(stemmed_tokens)

# HASIL AKHIR

print("\nHasil Preprocessing Final:")
print(stemmed_tokens)

Tweet awal:
Produk ini gak bagus!!! Sistemnya sering error 😭 #kecewa

1. Case Folding:
produk ini gak bagus!!! sistemnya sering error 😭 #kecewa

2. Cleaning Text:
produk ini gak bagus sistemnya sering error kecewa

3. Tokenization:
['produk', 'ini', 'gak', 'bagus', 'sistemnya', 'sering', 'error', 'kecewa']

4. Normalization:
['produk', 'ini', 'tidak', 'bagus', 'sistemnya', 'sering', 'error', 'kecewa']

5. Negation Handling:
['produk', 'ini', 'tidak', 'bagus_NEG', 'sistemnya', 'sering', 'error', 'kecewa']

6. Stopword Removal:
['produk', 'tidak', 'bagus_NEG', 'sistemnya', 'sering', 'error', 'kecewa']

7. Stemming:
['produk', 'tidak', 'bagus_NEG', 'sistem', 'sering', 'error', 'kecewa']

Hasil Preprocessing Final:
['produk', 'tidak', 'bagus_NEG', 'sistem', 'sering', 'error', 'kecewa']


### Contoh Hasil Pipeline

Tweet awal:

> “Produk ini gak bagus!!! Sistemnya sering error 😭 #kecewa”

Hasil preprocessing:

> [“produk”, “tidak”, “bagus_NEG”, “sistem”, “sering”, “error”, “kecewa”]

Pipeline ini cocok untuk sentiment analysis tweet bahasa Indonesia karena tetap mempertahankan informasi sentimen dan mengurangi noise dari media sosial.

## 5. Jelaskan perbedaan stemming dan lemmatization dengan contoh dari bahasa Indonesia.

Stemming dan lemmatization merupakan dua teknik preprocessing NLP yang digunakan untuk mengubah kata menjadi bentuk dasar. Namun, keduanya memiliki pendekatan yang berbeda sehingga hasilnya juga dapat berbeda.


In [7]:
ID_SIMULATED_LEMMA = {
    "membaca": "baca",
    "dibaca": "baca",
    "pembacaan": "baca",
    "bermain": "main",
    "permainan": "main",
    "pemain": "pemain",      # dipertahankan karena nomina pelaku
    "pembangunan": "bangun",
    "pengembangan": "kembang",
    "melakukan": "laku",
}

def simulated_lemmatize_id(words: List[str]) -> List[str]:
    return [ID_SIMULATED_LEMMA.get(w, w) for w in words]

words = ["membaca", "pembangunan", "pengembangan", "berjalan", "makanan", "permainan", "pemerintah"]
rows = []
for w, lemma in zip(words, simulated_lemmatize_id(words)):
    rows.append([w, stemmer.stem(w), lemma])

pd.DataFrame(rows, columns=["kata", "stemming", "simulasi_lemmatization"])

,kata,stemming,simulasi_lemmatization
0,membaca,baca,baca
1,pembangunan,bangun,bangun
2,pengembangan,kembang,kembang
3,berjalan,jalan,berjalan
4,makanan,makan,makanan
5,permainan,main,main
6,pemerintah,perintah,pemerintah


Berdasarkan contoh tersebut, terlihat bahwa stemming cenderung memotong imbuhan secara langsung, sedangkan lemmatization mempertimbangkan makna dan fungsi kata. 

Pada kata 'membaca', 'pembangunan', dan 'pengembangan', hasil stemming dan lemmatization sama karena keduanya memang dapat direduksi ke bentuk dasar yang sesuai, yaitu 'baca', 'bangun', dan 'kembang'.

Namun, perbedaan mulai terlihat pada beberapa kata berikut.

* 'berjalan'

  * stemming → 'jalan'
  * lemmatization → 'berjalan'

  Stemming menghapus prefiks *ber-* sehingga hanya menyisakan akar kata. Lemmatization mempertahankan bentuk 'berjalan' karena kata tersebut masih dianggap bentuk valid secara linguistik.

* 'makanan'

  * stemming → 'makan'
  * lemmatization → 'makanan'

  Pada stemming, sufiks *-an* dihapus sehingga kata berubah menjadi 'makan'. Lemmatization mempertahankan 'makanan' karena kata tersebut memiliki makna khusus sebagai nomina, bukan sekadar bentuk turunan dari kata kerja 'makan'.

* 'pemerintah'

  * stemming → 'perintah'
  * lemmatization → 'pemerintah'

  Ini menunjukkan risiko *over-stemming*. Kata 'pemerintah' memiliki makna berbeda dengan 'perintah', tetapi stemming menghapus imbuhan sehingga keduanya dianggap sama. Lemmatization mempertahankan bentuk 'pemerintah' agar makna tetap terjaga.

Dari contoh tersebut dapat disimpulkan bahwa:

* stemming lebih sederhana dan fokus pada pemotongan imbuhan,
* lemmatization lebih memperhatikan konteks linguistik dan makna kata.

Karena itu, stemming biasanya lebih cepat dan ringan, sedangkan lemmatization lebih baik untuk tugas NLP yang membutuhkan ketepatan semantik.

# 03

## Latihan 1 — Manual TF-IDF

Diberikan corpus:

```text
D1: data data analisis sistem
D2: data mining sistem
D3: bahasa alami sistem
```

Hitung manual untuk term `data`:

1. TF raw pada setiap dokumen.
2. DF.
3. IDF klasik.
4. TF-IDF raw × IDF klasik.

### 1. TF Raw

TF (Term Frequency) raw adalah jumlah kemunculan term dalam dokumen.

| Dokumen | Isi Dokumen               | TF(data) |
| ------- | ------------------------- | -------- |
| D1      | data data analisis sistem | 2        |
| D2      | data mining sistem        | 1        |
| D3      | bahasa alami sistem       | 0        |

Jadi:

* TF(data, D1) = 2
* TF(data, D2) = 1
* TF(data, D3) = 0

---

### 2. DF (Document Frequency)

DF adalah jumlah dokumen yang mengandung term.

Term `data` muncul pada:

* D1
* D2

Tidak muncul pada D3.

Maka:

$$
DF(data) = 2
$$

---

### 3. IDF Klasik

Rumus IDF klasik:

$$
IDF(t)=\log\left(\frac{N}{DF(t)}\right)
$$

Dengan:

* N = 3 (jumlah dokumen)
* DF(data)=2

Maka:

$$
IDF(data)=\log\left(\frac{3}{2}\right)
$$

Jika menggunakan log basis 10:

$$
IDF(data)=\log(1.5)\approx 0.176
$$

---

### 4. TF-IDF = TF × IDF

Rumus:

$$
TF\text{-}IDF(t,d)=TF(t,d)\times IDF(t)
$$

Karena:

$$
IDF(data) \approx 0.176
$$

Maka:

| Dokumen | TF(data) | IDF(data) | TF-IDF |
| ------- | -------- | --------- | ------ |
| D1      | 2        | 0.176     | 0.352  |
| D2      | 1        | 0.176     | 0.176  |
| D3      | 0        | 0.176     | 0      |

## Latihan 2 — Variasi TF

Gunakan dokumen:

```text
data data data analisis sistem
```

Hitung untuk term `data`:

1. raw TF,
2. binary TF,
3. normalized TF,
4. log TF,
5. augmented TF.

Jelaskan perbedaan interpretasinya.

### 1. Raw TF

Raw TF adalah jumlah kemunculan langsung suatu term dalam dokumen.

Rumus:

$$
TF_{raw}(t,d)=f(t,d)
$$

Dengan:

* f(t,d) = frekuensi term (t) pada dokumen (d)

Karena term `data` muncul sebanyak 3 kali, maka:

$$
TF_{raw}(data,d)=3
$$

---

### 2. Binary TF

Binary TF hanya melihat apakah term muncul atau tidak dalam dokumen.

Rumus:

$$
TF_{binary}(t,d) =
\begin{cases}
1, & \text{jika } f(t,d) > 0 \\
0, & \text{jika } f(t,d) = 0
\end{cases}
$$

Karena `data` muncul dalam dokumen, maka:

$$
TF_{binary}(data,d)=1
$$

---

### 3. Normalized TF

Normalized TF membagi frekuensi term dengan total jumlah kata dalam dokumen. Metode ini mengurangi bias akibat panjang dokumen.

Rumus:

$$
TF_{norm}(t,d)=\frac{f(t,d)}{|d|}
$$

Dengan:

* f(data,d)=3
* |d|=5 (sebagai jumlah total kata dalam dokumen)

Maka:

$$
TF_{norm}(data,d)=\frac{3}{5}=0.6
$$

---

### 4. Log TF

Log TF digunakan untuk menekan dominasi term yang terlalu sering muncul.

Rumus:

$$
TF_{log}(t,d) =
\begin{cases}
1 + \log(f(t,d)), & \text{jika } f(t,d) > 0 \\
0, & \text{jika } f(t,d) = 0
\end{cases}
$$

Dengan:

* f(data,d)=3

Maka:

$$
TF_{log}(data,d)=1+\log(3)
$$

$$
TF_{log}(data,d)=1+0.477=1.477
$$

---

### 5. Augmented TF

Augmented TF membandingkan frekuensi term dengan term paling sering di dokumen tersebut.

Rumus:

$$
TF_{aug}(t,d) = 0.5 + 0.5 \times \frac{f(t,d)}{max_f(d)}
$$

Dengan:

* f(data,d)=3)
* frekuensi maksimum dalam dokumen = 3

Maka:

$$
TF_{aug}(data,d)=0.5+0.5\times\frac{3}{3}
$$

$$
TF_{aug}(data,d)=1
$$

---

### Ringkasan Hasil

| Jenis TF      | Nilai | Interpretasi                                                           |
| ------------- | ----- | ---------------------------------------------------------------------- |
| Raw TF        | 3     | Menghitung jumlah kemunculan asli term dalam dokumen.                  |
| Binary TF     | 1     | Hanya melihat keberadaan term, bukan jumlahnya.                        |
| Normalized TF | 0.6   | Mengukur proporsi term terhadap panjang dokumen.                       |
| Log TF        | 1.477 | Mengurangi pengaruh kata yang terlalu sering muncul.                   |
| Augmented TF  | 1     | Membandingkan frekuensi term dengan term paling dominan dalam dokumen. |

## Latihan 3 — Eksperimen `ngram_range`

Gunakan corpus minimal 10 dokumen Bahasa Indonesia. Bandingkan:

```python
ngram_range=(1,1)
ngram_range=(1,2)
ngram_range=(1,3)
```

Laporkan:

1. jumlah fitur,
2. contoh fitur,
3. sparsity matrix,
4. top terms per dokumen.

### 1. Jumlah Fitur

In [8]:
corpus = [
    "sistem informasi akademik kampus",
    "analisis data mahasiswa",
    "machine learning untuk analisis data",
    "pemrosesan bahasa alami indonesia",
    "analisis data multivariat",
    "visualisasi data penelitian",
    "algoritma klasifikasi teks",
    "data mining dan machine learning",
    "pengolahan teks bahasa indonesia",
    "analisis sentimen media sosial"
]

def print_section(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def tokenize(text: str) -> List[str]:
    return re.findall(r"(?u)\b\w+\b", text.lower())

def show_df(df: pd.DataFrame, title: str | None = None):
    if title:
        print_section(title)
    display(df)

def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def stringify_ngrams(grams: List[Tuple[str, ...]]) -> List[str]:
    return [" ".join(g) for g in grams]

def vocab_for_ngram_range(corpus: List[str], ngram_range: Tuple[int, int]) -> List[str]:
    min_n, max_n = ngram_range
    vocab_ngram = set()
    for doc in corpus:
        toks = tokenize(doc)
        for n in range(min_n, max_n + 1):
            vocab_ngram.update(stringify_ngrams(make_ngrams(toks, n)))
    return sorted(vocab_ngram)

rows = []
for ngram_range in [(1,1), (1,2), (1,3)]:
    features = vocab_for_ngram_range(corpus, ngram_range)
    rows.append({
        "ngram_range": str(ngram_range),
        "jumlah_fitur": len(features),
    })

ngram_growth_df = pd.DataFrame(rows)
show_df(ngram_growth_df, "Perbandingan Jumlah Fitur Berdasarkan Variasi ngram_range")


Perbandingan Jumlah Fitur Berdasarkan Variasi ngram_range


,ngram_range,jumlah_fitur
0,"(1, 1)",26
1,"(1, 2)",51
2,"(1, 3)",69


### 2. Contoh Fitur

In [9]:
corpus = [
    "sistem informasi akademik kampus",
    "analisis data mahasiswa",
    "machine learning untuk analisis data",
    "pemrosesan bahasa alami indonesia",
    "analisis data multivariat",
    "visualisasi data penelitian",
    "algoritma klasifikasi teks",
    "data mining dan machine learning",
    "pengolahan teks bahasa indonesia",
    "analisis sentimen media sosial"
]

def print_section(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def tokenize(text: str) -> List[str]:
    return re.findall(r"(?u)\b\w+\b", text.lower())

def show_df(df: pd.DataFrame, title: str | None = None):
    if title:
        print_section(title)
    display(df)

def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def stringify_ngrams(grams: List[Tuple[str, ...]]) -> List[str]:
    return [" ".join(g) for g in grams]

def vocab_for_ngram_range(corpus: List[str], ngram_range: Tuple[int, int]) -> List[str]:
    min_n, max_n = ngram_range
    vocab_ngram = set()
    for doc in corpus:
        toks = tokenize(doc)
        for n in range(min_n, max_n + 1):
            vocab_ngram.update(stringify_ngrams(make_ngrams(toks, n)))
    return sorted(vocab_ngram)

rows = []
for ngram_range in [(1,1), (1,2), (1,3)]:
    features = vocab_for_ngram_range(corpus, ngram_range)
    rows.append({
        "ngram_range": str(ngram_range),
        "contoh_fitur": ", ".join(features[:10])
    })

ngram_growth_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', None)
show_df(ngram_growth_df, "Perbandingan Contoh Fitur Berdasarkan Variasi ngram_range")


Perbandingan Contoh Fitur Berdasarkan Variasi ngram_range


,ngram_range,contoh_fitur
0,"(1, 1)","akademik, alami, algoritma, analisis, bahasa, dan, data, indonesia, informasi, kampus"
1,"(1, 2)","akademik, akademik kampus, alami, alami indonesia, algoritma, algoritma klasifikasi, analisis, analisis data, analisis sentimen, bahasa"
2,"(1, 3)","akademik, akademik kampus, alami, alami indonesia, algoritma, algoritma klasifikasi, algoritma klasifikasi teks, analisis, analisis data, analisis data mahasiswa"


### 3. Sparsity Matrix

Rumus sparsity:

$$
Sparsity=1-\frac{\text{jumlah elemen nonzero}}{\text{jumlah total elemen}}
$$



In [10]:
rows = []

for ngram_range in [(1,1), (1,2), (1,3)]:

    vectorizer = CountVectorizer(
        ngram_range=ngram_range
    )

    X = vectorizer.fit_transform(corpus)

    nonzero = X.nnz

    total_elements = X.shape[0] * X.shape[1]

    sparsity = 1 - (nonzero / total_elements)

    rows.append({
        "ngram_range": str(ngram_range),
        "shape_matrix": str(X.shape),
        "nonzero": nonzero,
        "total_elements": total_elements,
        "sparsity": round(sparsity, 4)
    })

sparsity_df = pd.DataFrame(rows)

show_df(
    sparsity_df,
    "Perbandingan Sparsity Matrix untuk Berbagai ngram_range"
)


Perbandingan Sparsity Matrix untuk Berbagai ngram_range


,ngram_range,shape_matrix,nonzero,total_elements,sparsity
0,"(1, 1)","(10, 26)",38,260,0.8538
1,"(1, 2)","(10, 51)",66,510,0.8706
2,"(1, 3)","(10, 69)",84,690,0.8783


### 4. Top Terms per Dokumen

In [11]:
doc_ids = [f"D{i+1}" for i in range(len(corpus))]

def top_terms_per_doc(vectorizer, X, doc_ids: List[str], top_k: int = 5) -> pd.DataFrame:
    features = np.array(vectorizer.get_feature_names_out())
    arr = X.toarray()
    rows = []
    for i, doc_id in enumerate(doc_ids):
        top_idx = np.argsort(arr[i])[::-1][:top_k]
        rows.append({
            "dokumen": doc_id,
            "teks": corpus[i] if i < len(corpus) else "",
            "top_terms": ", ".join([f"{features[j]} ({arr[i,j]:.4f})" for j in top_idx if arr[i,j] > 0])
        })
    return pd.DataFrame(rows)

vec = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1,2),
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True,
    norm="l2"
)
X = vec.fit_transform(corpus)
show_df(top_terms_per_doc(vec, X, doc_ids, top_k=7), "Top Terms per Dokumen dengan TF-IDF")


Top Terms per Dokumen dengan TF-IDF


,dokumen,teks,top_terms
0,D1,sistem informasi akademik kampus,"sistem (0.3780), sistem informasi (0.3780), informasi akademik (0.3780), akademik (0.3780), kampus (0.3780), informasi (0.3780), akademik kampus (0.3780)"
1,D2,analisis data mahasiswa,"mahasiswa (0.5469), data mahasiswa (0.5469), analisis data (0.4068), analisis (0.3616), data (0.3248)"
2,D3,machine learning untuk analisis data,"untuk analisis (0.3919), untuk (0.3919), learning untuk (0.3919), learning (0.3332), machine (0.3332), machine learning (0.3332), analisis data (0.2915)"
3,D4,pemrosesan bahasa alami indonesia,"pemrosesan (0.3939), pemrosesan bahasa (0.3939), alami (0.3939), alami indonesia (0.3939), bahasa alami (0.3939), indonesia (0.3348), bahasa (0.3348)"
4,D5,analisis data multivariat,"multivariat (0.5469), data multivariat (0.5469), analisis data (0.4068), analisis (0.3616), data (0.3248)"
5,D6,visualisasi data penelitian,"visualisasi data (0.4793), visualisasi (0.4793), penelitian (0.4793), data penelitian (0.4793), data (0.2846)"
6,D7,algoritma klasifikasi teks,"algoritma (0.4602), klasifikasi (0.4602), klasifikasi teks (0.4602), algoritma klasifikasi (0.4602), teks (0.3912)"
7,D8,data mining dan machine learning,"mining (0.3646), mining dan (0.3646), data mining (0.3646), dan (0.3646), dan machine (0.3646), machine learning (0.3100), learning (0.3100)"
8,D9,pengolahan teks bahasa indonesia,"pengolahan (0.4027), pengolahan teks (0.4027), teks bahasa (0.4027), bahasa indonesia (0.4027), teks (0.3423), indonesia (0.3423), bahasa (0.3423)"
9,D10,analisis sentimen media sosial,"sentimen media (0.3941), sentimen (0.3941), sosial (0.3941), media (0.3941), analisis sentimen (0.3941), media sosial (0.3941), analisis (0.2606)"


## Latihan 4 — Eksperimen `min_df` dan `max_df`

Bandingkan konfigurasi:

```python
min_df=1
min_df=2
max_df=1.0
max_df=0.8
```

Jelaskan term apa saja yang hilang dan mengapa.

### 1. Konsep `min_df` dan `max_df`

#### `min_df`

`min_df` membuang term yang muncul di terlalu sedikit dokumen.

Contoh:

```python
min_df=2
```

berarti term harus muncul minimal di 2 dokumen.

---

#### `max_df`

`max_df` membuang term yang muncul di terlalu banyak dokumen.

Contoh:

```python
max_df=0.8
```

berarti term yang muncul di lebih dari 80% dokumen akan dibuang.

### 2. Eksperimen `min_df=1`

In [12]:
vectorizer = CountVectorizer(min_df=1)

X = vectorizer.fit_transform(corpus)

fitur = vectorizer.get_feature_names_out()

print(fitur)

['akademik' 'alami' 'algoritma' 'analisis' 'bahasa' 'dan' 'data'
 'indonesia' 'informasi' 'kampus' 'klasifikasi' 'learning' 'machine'
 'mahasiswa' 'media' 'mining' 'multivariat' 'pemrosesan' 'penelitian'
 'pengolahan' 'sentimen' 'sistem' 'sosial' 'teks' 'untuk' 'visualisasi']


Semua term dipertahankan karena syarat minimal kemunculan = 1 dokumen.

### 3. Eksperimen `min_df=2`

In [13]:
vectorizer = CountVectorizer(min_df=2)

X = vectorizer.fit_transform(corpus)

fitur = vectorizer.get_feature_names_out()

print(fitur)

['analisis' 'bahasa' 'data' 'indonesia' 'learning' 'machine' 'teks']


Pada konfigurasi:

```python id="slbqfa"
min_df=2
```

Scikit-learn hanya mempertahankan term yang muncul minimal pada 2 dokumen.

Secara matematis:

$$
DF(term) \ge 2
$$

Jika:

$$
DF(term) < 2
$$

maka term dihapus dari vocabulary.

### 4. Eksperimen `max_df=1.0`

In [14]:
vectorizer = CountVectorizer(max_df=1.0)

X = vectorizer.fit_transform(corpus)

fitur = vectorizer.get_feature_names_out()

print(fitur)

['akademik' 'alami' 'algoritma' 'analisis' 'bahasa' 'dan' 'data'
 'indonesia' 'informasi' 'kampus' 'klasifikasi' 'learning' 'machine'
 'mahasiswa' 'media' 'mining' 'multivariat' 'pemrosesan' 'penelitian'
 'pengolahan' 'sentimen' 'sistem' 'sosial' 'teks' 'untuk' 'visualisasi']


Tidak ada term yang dihapus. Karena `1.0` berarti term boleh muncul sampai 100% dokumen, maka semua term tetap dipakai.

### 5. Eksperimen `max_df=0.8`

In [15]:
vectorizer = CountVectorizer(max_df=0.8)

X = vectorizer.fit_transform(corpus)

fitur = vectorizer.get_feature_names_out()

print(fitur)

['akademik' 'alami' 'algoritma' 'analisis' 'bahasa' 'dan' 'data'
 'indonesia' 'informasi' 'kampus' 'klasifikasi' 'learning' 'machine'
 'mahasiswa' 'media' 'mining' 'multivariat' 'pemrosesan' 'penelitian'
 'pengolahan' 'sentimen' 'sistem' 'sosial' 'teks' 'untuk' 'visualisasi']


Output `max_df=0.8` sama dengan `max_df=1.0` karena tidak ada term yang muncul di lebih dari 80% dokumen.

Jumlah dokumen:

$$
N=10
$$

Batas maksimum:

$$
0.8 \times 10 = 8
$$

Artinya term akan dihapus jika muncul pada lebih dari 8 dokumen.

Namun:

* term `data` hanya muncul pada 5 dokumen,
* term `analisis` hanya muncul pada 4 dokumen.

Karena tidak ada term dengan DF > 8, maka tidak ada term yang hilang pada `max_df=0.8`.

### 6. Term yang Hilang

Term berikut hilang saat menggunakan `min_df=2`:

| Term        | Alasan                      |
| ----------- | --------------------------- |
| akademik    | hanya muncul pada 1 dokumen |
| alami       | hanya muncul pada 1 dokumen |
| algoritma   | hanya muncul pada 1 dokumen |
| dan         | hanya muncul pada 1 dokumen |
| informasi   | hanya muncul pada 1 dokumen |
| kampus      | hanya muncul pada 1 dokumen |
| klasifikasi | hanya muncul pada 1 dokumen |
| mahasiswa   | hanya muncul pada 1 dokumen |
| media       | hanya muncul pada 1 dokumen |
| mining      | hanya muncul pada 1 dokumen |
| multivariat | hanya muncul pada 1 dokumen |
| pemrosesan  | hanya muncul pada 1 dokumen |
| penelitian  | hanya muncul pada 1 dokumen |
| pengolahan  | hanya muncul pada 1 dokumen |
| sentimen    | hanya muncul pada 1 dokumen |
| sistem      | hanya muncul pada 1 dokumen |
| sosial      | hanya muncul pada 1 dokumen |
| untuk       | hanya muncul pada 1 dokumen |
| visualisasi | hanya muncul pada 1 dokumen |

---

### Term yang Tetap Dipertahankan

Karena muncul minimal pada 2 dokumen:

| Term      | Muncul pada Dokumen |
| --------- | ------------------- |
| analisis  | D2, D3, D5, D10     |
| bahasa    | D4, D9              |
| data      | D2, D3, D5, D6, D8  |
| indonesia | D4, D9              |
| learning  | D3, D8              |
| machine   | D3, D8              |
| teks      | D7, D9              |

## Latihan 5 — Mini Search Engine

Buat mini search engine TF-IDF dengan minimal 20 dokumen.

Untuk setiap query, tampilkan 5 dokumen paling relevan berdasarkan cosine similarity.

Langkah membuat mini search engine TF-IDF:

1. Fit `TfidfVectorizer` pada corpus.
2. Transform query menjadi vektor dengan vocabulary yang sama.
3. Hitung cosine similarity antara query dan setiap dokumen.
4. Urutkan dokumen berdasarkan skor similarity.


In [16]:
# Corpus Dokumen

documents = [
    "analisis data mahasiswa menggunakan python",
    "machine learning untuk klasifikasi data",
    "pemrosesan bahasa alami pada twitter",
    "visualisasi data penelitian akademik",
    "sistem informasi akademik kampus",
    "algoritma naive bayes untuk klasifikasi teks",
    "data mining pada sistem rekomendasi",
    "deep learning untuk computer vision",
    "analisis sentimen media sosial indonesia",
    "pengolahan teks bahasa indonesia",
    "clustering data menggunakan kmeans",
    "regresi linear untuk prediksi penjualan",
    "transformer model pada natural language processing",
    "text preprocessing pada data twitter",
    "analisis data multivariat",
    "cosine similarity pada pencarian dokumen",
    "tfidf untuk representasi teks",
    "klasifikasi berita menggunakan machine learning",
    "topic modeling pada kumpulan dokumen",
    "sistem pencarian informasi berbasis tfidf"
]

# Membuat ID Dokumen

doc_ids = [f"D{i+1}" for i in range(len(documents))]

# TF-IDF Vectorization

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    norm="l2"
)

X = vectorizer.fit_transform(documents)

features = vectorizer.get_feature_names_out()

# Cosine Similarity Antar Dokumen

sim = cosine_similarity(X)

sim_df = pd.DataFrame(
    sim,
    index=doc_ids,
    columns=doc_ids
)

In [17]:
# Dokumen Paling Mirip

print("Dokumen paling mirip untuk setiap dokumen:\n")

for i, doc_id in enumerate(doc_ids):

    scores = sim[i].copy()

    # Hilangkan similarity dengan dirinya sendiri
    scores[i] = -1

    j = int(np.argmax(scores))

    print(
        f"{doc_id} paling mirip dengan "
        f"{doc_ids[j]} "
        f"| score={scores[j]:.4f}"
    )

# Fungsi Search Engine

def search(
    query: str,
    vectorizer,
    X,
    documents: List[str],
    doc_ids: List[str],
    top_k: int = 5
) -> pd.DataFrame:

    # Transform query ke vector TF-IDF
    q_vec = vectorizer.transform([query])

    # Hitung cosine similarity
    scores = cosine_similarity(q_vec, X).ravel()

    # Ranking similarity terbesar
    order = np.argsort(scores)[::-1]

    results = []

    for i in order:

        # Hanya tampilkan similarity > 0
        if scores[i] > 0:

            results.append({
                "rank": len(results) + 1,
                "doc_id": doc_ids[i],
                "score": scores[i],
                "document": documents[i]
            })

    # Ambil top-k dokumen
    return pd.DataFrame(results[:top_k])

# Query Retrieval

queries = [
    "analisis data",
    "data learning",
    "sistem data",
    "data mining",
]

# Menampilkan Hasil Retrieval

for q in queries:

    print(f"\nQuery: {q}")

    hasil = search(
        q,
        vectorizer,
        X,
        documents,
        doc_ids
    )

    display(hasil)

Dokumen paling mirip untuk setiap dokumen:

D1 paling mirip dengan D15 | score=0.3650
D2 paling mirip dengan D18 | score=0.6314
D3 paling mirip dengan D14 | score=0.3134
D4 paling mirip dengan D5 | score=0.2457
D5 paling mirip dengan D20 | score=0.3960
D6 paling mirip dengan D2 | score=0.2963
D7 paling mirip dengan D14 | score=0.2142
D8 paling mirip dengan D2 | score=0.3183
D9 paling mirip dengan D15 | score=0.2135
D10 paling mirip dengan D3 | score=0.2185
D11 paling mirip dengan D1 | score=0.2973
D12 paling mirip dengan D2 | score=0.1274
D13 paling mirip dengan D7 | score=0.0921
D14 paling mirip dengan D3 | score=0.3134
D15 paling mirip dengan D1 | score=0.3650
D16 paling mirip dengan D19 | score=0.2876
D17 paling mirip dengan D6 | score=0.2948
D18 paling mirip dengan D2 | score=0.6314
D19 paling mirip dengan D16 | score=0.2876
D20 paling mirip dengan D5 | score=0.3960

Query: analisis data


,rank,doc_id,score,document
0,1,D15,0.702240,analisis data multivariat
1,2,D1,0.519716,analisis data mahasiswa menggunakan python
2,3,D9,0.304006,analisis sentimen media sosial indonesia
3,4,D2,0.207321,machine learning untuk klasifikasi data
4,5,D11,0.202153,clustering data menggunakan kmeans



Query: data learning


,rank,doc_id,score,document
0,1,D2,0.586694,machine learning untuk klasifikasi data
1,2,D18,0.333390,klasifikasi berita menggunakan machine learning
2,3,D8,0.315729,deep learning untuk computer vision
3,4,D15,0.248151,analisis data multivariat
4,5,D11,0.202153,clustering data menggunakan kmeans



Query: sistem data


,rank,doc_id,score,document
0,1,D7,0.537707,data mining pada sistem rekomendasi
1,2,D5,0.357982,sistem informasi akademik kampus
2,3,D20,0.321038,sistem pencarian informasi berbasis tfidf
3,4,D15,0.248151,analisis data multivariat
4,5,D2,0.207321,machine learning untuk klasifikasi data



Query: data mining


,rank,doc_id,score,document
0,1,D7,0.631935,data mining pada sistem rekomendasi
1,2,D15,0.211149,analisis data multivariat
2,3,D2,0.176407,machine learning untuk klasifikasi data
3,4,D11,0.172010,clustering data menggunakan kmeans
4,5,D4,0.168002,visualisasi data penelitian akademik


### Interpretasi Cosine Similarity Antar Dokumen

Hasil tersebut menunjukkan dokumen yang paling mirip berdasarkan representasi TF-IDF dan cosine similarity. Semakin besar score cosine similarity, maka semakin mirip isi dokumen.

Rumus cosine similarity:

$$
cosine(A,B) = \frac{A \cdot B}{\|A\|\|B\|}
$$

Nilai similarity:

* mendekati 1 → sangat mirip,
* mendekati 0 → tidak mirip.

---

#### 1. Interpretasi Dokumen Paling Mirip

##### D2 paling mirip dengan D18 | score = 0.6314

```text id="qebmpm"
D2 = machine learning untuk klasifikasi data
D18 = klasifikasi berita menggunakan machine learning
```

Interpretasi:

* Kedua dokumen memiliki term penting:

  * machine learning,
  * klasifikasi.
* Karena banyak term yang sama, similarity menjadi tinggi.
* Score 0.6314 menunjukkan hubungan topik yang kuat.

##### D1 paling mirip dengan D15 | score = 0.3650

```text id="mlfwok"
D1 = analisis data mahasiswa menggunakan python
D15 = analisis data multivariat
```

Interpretasi:

* Kedua dokumen memiliki term:

  * analisis,
  * data.
* TF-IDF menganggap kedua dokumen berada pada topik analisis data.
* Similarity sedang karena konteks akhirnya berbeda:

  * mahasiswa,
  * multivariat.

##### D3 paling mirip dengan D14 | score = 0.3134

```text id="yscqdz"
D3 = pemrosesan bahasa alami pada twitter
D14 = text preprocessing pada data twitter
```

Interpretasi:

* Sama-sama membahas preprocessing dan data Twitter.
* Memiliki term:

  * twitter,
  * pemrosesan/preprocessing.
* Similarity cukup tinggi karena topik NLP serupa.

##### D20 paling mirip dengan D5 | score = 0.3960

```text id="rhivbx"
D20 = sistem pencarian informasi berbasis tfidf
D5 = sistem informasi akademik kampus
```

Interpretasi:

* Kedua dokumen memiliki term:

  * sistem,
  * informasi.
* Walaupun konteks berbeda, TF-IDF tetap menganggap keduanya cukup mirip karena term penting overlap.

##### D13 paling mirip dengan D7 | score = 0.0921

```text id="glppvu"
D13 = transformer model pada natural language processing
D7 = data mining pada sistem rekomendasi
```

Interpretasi:

* Similarity sangat kecil.
* Hanya sedikit term yang sama, misalnya:

  * pada.
* Secara topik sebenarnya berbeda.
* Nilai kecil menunjukkan hubungan lemah.

---

#### 2. Interpretasi Query Retrieval

##### Query: `analisis data`

Interpretasi:

* Query mengandung:

  * analisis,
  * data.
* D15 paling relevan karena mengandung kedua term secara langsung:

```text id="wwblmn"
analisis data multivariat
```

* D1 juga relevan karena memiliki:

```text id="tqhnqv"
analisis data mahasiswa
```

* Dokumen lain memiliki score lebih kecil karena hanya mengandung sebagian term.

##### Query: `data learning`

Interpretasi:

* D2 paling relevan karena memiliki:

  * data,
  * learning.
* D18 relevan karena mengandung:

  * machine learning.
* D8 juga relevan karena mengandung:

  * deep learning.

TF-IDF menganggap kata `learning` sebagai term penting dalam query.

##### Query: `sistem data`

Interpretasi:

* D7 paling relevan karena mengandung:

```text id="hyjzjw"
data mining pada sistem rekomendasi
```

* D5 relevan karena memiliki term:

  * sistem,
  * informasi.
* D20 relevan karena memiliki:

  * sistem,
  * informasi.

Dokumen yang memiliki kedua kata query memperoleh score lebih tinggi.

##### Query: `data mining`

Interpretasi:

* D7 menjadi paling relevan karena frasa:

```text id="jlwmfq"
data mining
```

muncul langsung pada dokumen.

* Dokumen lain hanya memiliki kata `data` tanpa `mining`, sehingga similarity lebih rendah.

## Latihan 6 — Analisis Akademik

Tuliskan analisis 1–2 halaman:

> Bagaimana perubahan `ngram_range`, `min_df`, `max_df`, `sublinear_tf`, `smooth_idf`, dan `norm` memengaruhi representasi teks dan hasil retrieval?

Representasi teks merupakan tahapan penting dalam Natural Language Processing (NLP), khususnya pada sistem information retrieval dan search engine berbasis TF-IDF. Kualitas representasi teks sangat dipengaruhi oleh parameter yang digunakan pada proses vectorization. Parameter seperti `ngram_range`, `min_df`, `max_df`, `sublinear_tf`, `smooth_idf`, dan `norm` memiliki pengaruh langsung terhadap jumlah fitur, bobot kata, sparsity matrix, serta hasil retrieval dokumen.

Parameter `ngram_range` menentukan ukuran n-gram yang digunakan dalam representasi teks. Ketika menggunakan `ngram_range=(1,1)`, sistem hanya menggunakan unigram atau kata tunggal. Pendekatan ini menghasilkan jumlah fitur yang relatif sedikit dan proses komputasi lebih cepat. Namun, konteks antar kata belum dapat ditangkap dengan baik. Sebagai contoh, frasa “machine learning” akan dipisahkan menjadi dua term berbeda, yaitu “machine” dan “learning”. Akibatnya, retrieval terkadang kurang akurat karena sistem hanya memahami keberadaan kata secara individual. Ketika parameter diubah menjadi `ngram_range=(1,2)` atau `(1,3)`, sistem mulai mempertimbangkan bigram dan trigram. Frasa seperti “analisis data”, “machine learning”, atau “bahasa alami indonesia” dapat diperlakukan sebagai satu fitur utuh. Hal ini meningkatkan kemampuan sistem dalam memahami konteks dokumen dan query. Akan tetapi, semakin besar nilai n-gram, jumlah fitur meningkat secara signifikan sehingga matrix menjadi lebih sparse dan kebutuhan memori bertambah.

Selain `ngram_range`, parameter `min_df` juga memengaruhi representasi teks. Parameter ini menentukan jumlah minimum dokumen yang harus mengandung suatu term agar term tersebut dipertahankan dalam vocabulary. Ketika menggunakan `min_df=1`, semua term akan dipakai, termasuk kata yang sangat jarang muncul. Kondisi ini dapat meningkatkan noise karena banyak term unik yang sebenarnya kurang informatif. Sebaliknya, ketika menggunakan `min_df=2` atau lebih besar, term yang hanya muncul pada satu dokumen akan dihapus. Penghapusan ini membantu mengurangi dimensi fitur dan meningkatkan efisiensi komputasi. Pada sistem retrieval, penggunaan `min_df` yang terlalu tinggi dapat menyebabkan hilangnya kata penting yang bersifat spesifik terhadap suatu dokumen. Oleh karena itu, pemilihan nilai `min_df` harus disesuaikan dengan ukuran corpus dan tujuan analisis.

Parameter `max_df` bekerja berlawanan dengan `min_df`. Parameter ini menghapus term yang terlalu sering muncul pada dokumen. Jika menggunakan `max_df=1.0`, semua kata tetap dipertahankan. Namun, ketika menggunakan `max_df=0.8`, term yang muncul pada lebih dari 80% dokumen akan dihapus dari vocabulary. Tujuan utama parameter ini adalah menghilangkan kata yang terlalu umum dan kurang informatif, mirip dengan konsep stopword removal. Dalam retrieval dokumen, kata yang terlalu sering muncul biasanya tidak membantu membedakan dokumen satu dengan lainnya. Oleh sebab itu, penggunaan `max_df` dapat meningkatkan kualitas retrieval dengan mempertahankan term yang lebih diskriminatif.

Parameter berikutnya adalah `sublinear_tf`. Parameter ini mengubah perhitungan TF dari bentuk linear menjadi logaritmik. Secara default, TF dihitung berdasarkan jumlah kemunculan term secara langsung. Masalahnya, kata yang muncul terlalu sering dapat mendominasi bobot dokumen. Ketika `sublinear_tf=True`, TF dihitung menggunakan transformasi logaritma sehingga pertumbuhan bobot menjadi lebih stabil. Pendekatan ini membantu mengurangi dominasi term yang sangat sering muncul dalam satu dokumen. Dalam retrieval, penggunaan sublinear TF sering menghasilkan ranking dokumen yang lebih seimbang karena sistem tidak terlalu bias terhadap frekuensi tinggi.

Parameter `smooth_idf` digunakan untuk menghindari pembagian nol pada perhitungan IDF. Ketika `smooth_idf=True`, sistem menambahkan nilai satu pada jumlah dokumen dan document frequency sebelum menghitung IDF. Teknik smoothing ini membuat nilai IDF lebih stabil, terutama pada corpus kecil. Tanpa smoothing, term yang muncul pada seluruh dokumen dapat menghasilkan nilai IDF nol sehingga kontribusinya hilang sepenuhnya. Penggunaan `smooth_idf` membantu menjaga kestabilan representasi teks dan mengurangi risiko perhitungan ekstrem pada dataset terbatas.

Parameter terakhir adalah `norm`, yang digunakan untuk normalisasi vector TF-IDF. Normalisasi penting karena panjang dokumen sering kali berbeda. Jika `norm="l2"` digunakan, panjang vector akan dinormalisasi sehingga setiap dokumen memiliki panjang vector yang sama. Teknik ini sangat efektif ketika digunakan bersama cosine similarity karena similarity dihitung berdasarkan sudut antar vector, bukan panjang vector. Dengan normalisasi L2, dokumen panjang tidak otomatis memiliki skor lebih tinggi dibanding dokumen pendek. Sementara itu, jika normalisasi dimatikan (`norm=None`), dokumen dengan banyak kata cenderung memiliki bobot lebih besar sehingga retrieval menjadi bias terhadap panjang dokumen.

Secara keseluruhan, parameter pada TF-IDF memiliki pengaruh besar terhadap kualitas representasi teks dan hasil retrieval. Parameter `ngram_range` meningkatkan pemahaman konteks bahasa, `min_df` dan `max_df` membantu menyaring term yang tidak relevan, `sublinear_tf` menstabilkan bobot frekuensi term, `smooth_idf` menjaga kestabilan perhitungan IDF, dan `norm` memastikan perbandingan vector dilakukan secara adil. Kombinasi parameter yang tepat dapat meningkatkan akurasi retrieval, mengurangi noise, dan menghasilkan sistem pencarian dokumen yang lebih efektif serta efisien.
